# Suite ALG — The predicate algebra

`Predicates` extends mbse-expressions' Basic with terms about a store's data, each a data class with a builder:
quantifiers over a schema's objects (`Exists`, `Forall`, over an `extent`), predicates and their application, and the
terms of `Distributions` (suite DST).
`Contains` writes Basic's `any` over an adjacency's entries. Mandatory, possible and forbidden links are predicates.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, Distributions as D, Predicates as P, Queries as Q, Validators as V } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, equal, listed, raises, same } from "./support.js";

const [c, p, x] = [E.variable("c"), E.variable("p"), E.variable("x")];
const owns = P.Exists((q) => q.symbols({ p: Phone }).requires(P.Contains(c.phones, (e) => e.phone.eq(p))));
const every = P.Forall((q) => q.symbols({ c: Contact }).requires(owns)); // every contact has a phone
const none = P.Forall((q) => q.symbols({ c: Contact }).forbids(owns)); // no contact has one
const some = P.Forall((q) => q.symbols({ c: Contact }).requires(
  new D.Choices.Builder().arms((a) => a.weight(0.35).requires(owns),
    (a) => a.weight(0.65).requires(E.operation("not", owns))).create())); // 35% have one

## ALG-01 · The algebra's terms are built by their builders, and checked

In [ ]:
{
  assert(every instanceof P.OfForall && every.name === "c" && every.collection.schema === "Contact" && every.body === owns);
  assert(owns instanceof P.OfExists && owns.collection.schema === "Phone");
  assert(Terms.same(owns.body, E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p)).data));
  assert(Terms.same(none.body, E.operation("not", owns).data) && some.body.arms[0].weight === 0.35);
  assert(P.DIALECT.base === E.DIALECT && P.DIALECT.validate(every, { core: true }).length === 0 && P.DIALECT.validate(some).length === 0);
  assert(same([...P.DIALECT.kinds().keys()].filter((k) => !E.DIALECT.kinds().has(k)), [
    "extent", "forall", "exists", "predicate", "apply", "set", "choices", "arm", "count", "uniform", "normal", "poisson",
    "geometric", "categorical", "option"]));
  assert(same([...P.DIALECT.schemas.keys()].filter((n) => n.startsWith("Patterns.")).slice(0, 6), [
    "Patterns.OfExtent", "Patterns.OfForall", "Patterns.OfExists", "Patterns.Predicate", "Patterns.OfApply", "Patterns.Set"]));
  assert((E.DIALECT.validate(every)[0] as string).startsWith("not an expression: ")); // Basic alone cannot check it
  assert(same(P.DIALECT.validate(owns), ["body: collection: argument 0: variable 'c' is not bound"]));
  // Several symbols are quantified in turn; conditions add up, within them all.
  const pair = P.Exists((q) => q.symbols({ c: Contact }).symbols(new Map([["p", "Phone"]])).requires(
    P.Contains(c.phones, (e) => e.phone.eq(p))).forbids(p.has("number")));
  assert(pair.name === "c" && pair.body instanceof P.OfExists && pair.body.name === "p" && pair.body.collection.schema === "Phone");
  assert(Terms.same(pair.body.body, E.operation("and", owns.body, E.operation("not", p.has("number"))).data));
  const more = new P.OfExists.Builder(owns).requires(p.has("number")).clone() as P.OfExists; // conditions add to its body
  assert(Terms.same(more.body, E.operation("and", owns.body, p.has("number")).data) && more.collection === owns.collection);
  assert((new P.OfExists.Builder(more).clone() as P.OfExists).body === more.body && P.Exists(owns) === owns);
  assert(P.Exists((q) => q.symbols({ p: Phone })).body === null); // nothing validates yet
  assert(P.Exists((q) => q.requires(true)).name === null); // nor a quantifier without a symbol
  const numbered = P.Exists((q) => q.symbols({ p: Phone }).requires(q.p.has("number"))); // the builder gives its symbols
  assert(Terms.same(numbered.body, p.has("number").data));
  assert(new P.OfExists.Builder().p === undefined); // not declared yet: Python raises AttributeError
  assert(new P.OfExists.Builder(more).forbids(true).update() === more && more.body.name === "and");
  function written(e: any) {
    return e.phone.eq(p);
  }
  assert(Terms.same(P.Contains(c.phones, written), owns.body)); // a function, not only an arrow
  assert(new P.OfExtent.Builder().schema(Contact).create().schema === "Contact");
  raises(TypeError, () => P.Exists((q) => q.symbols({ p: new S.OfObject.Builder().create() })), "expected a named schema or its name, got ");
  raises(TypeError, () => P.Exists("p" as any), "expected an exists or a callable taking its builder, got 'p'");
  raises(TypeError, () => P.Forall("p" as any), "expected a forall or a callable taking its builder, got 'p'");
  for (const adjacency of [c, E.operation("get", c, 1n), c.has("phones"), E.operation("get", c, "phones", 1n)]) {
    raises(TypeError, () => P.Contains(adjacency, written), "Contains expects an object's adjacency, such as c.phones");
  }
  // A destructured parameter has no name to bind: TypeScript only, since a Python lambda cannot destructure.
  for (const condition of [() => true, (e: any, f: any) => true, (...e: any[]) => true, 3, ({ phone }: any) => phone.eq(p)]) {
    raises(TypeError, () => P.Contains(c.phones, condition as any), "Contains expects a function of one entry");
  }
}

## ALG-02 · The algebra is evaluated over the store

In [ ]:
{
  const store = book();
  const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((q: any) => q.number("1"))).create();
  const bob = store.Contact().name("bob").create();
  listed(store, ann, bob);
  const evaluate = new P.Evaluator(store);
  assert(evaluate.run(every) === false && evaluate.run(none) === false && evaluate.run(some) === true); // bob has none
  assert(evaluate.run(owns, { c: ann }) === true && evaluate.run(owns, { c: bob }) === false && evaluate.run(owns, { c: null }) === null);
  assert(evaluate.run(P.Forall((q) => q.symbols({ c: Contact }).requires(E.operation("or", owns, c.has("age").not_())))) === true);
  assert(evaluate.run(P.Exists((q) => q.symbols({ c: Contact }).requires(c.name.eq("bob")).forbids(owns))) === true);
  assert(same(evaluate.run(new P.OfExtent("Contact")) as unknown[], [ann, bob]));
  listed(store, store.Contact().name("cy").create());
  assert((evaluate.run(new P.OfExtent("Contact")) as unknown[]).length === 2
    && (new P.Evaluator(store).run(new P.OfExtent("Contact")) as unknown[]).length === 3); // read once each
  raises(Errors.AttributeError, () => evaluate.run(new P.OfExtent("Nope")), "no schema registered as 'Nope'");
  assert(evaluate.run(x, { x: true }) === true);
}

## ALG-03 · Predicates written in the algebra: about the whole store, through links, and as data

In [ ]:
{
  const store = book();
  const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((q: any) => q.number("1"))).create();
  const bob = store.Contact().name("bob").create();
  listed(store, ann, bob);
  const EveryoneHasAPhone = new P.OfPredicate.Builder().name("EveryoneHasAPhone").requires(every).create(); // no symbols
  assert(same(V.Validate(store, [EveryoneHasAPhone])(Contact, ann), ["the store: 'EveryoneHasAPhone' does not hold"]));
  const query = new Q.Scan(store);
  assert([...query.select(EveryoneHasAPhone)].length === 0 && same(query.explain(EveryoneHasAPhone), ["first 1 test"]));
  store.Contact(bob).phones((e: any) => e.phone((q: any) => q.number("2"))).update();
  assert(equal([...query.select(EveryoneHasAPhone)], [{}])); // one match, empty, when it holds
  const Numbered = new P.OfPredicate.Builder().name("Numbered").symbols({ c: Contact, p: Phone }).requires(
    P.Contains(c.phones, (e) => e.phone.eq(p))).requires(p.has("number")).create();
  assert(same(query.explain(Numbered), ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]));
  assert(equal([...query.select(Numbered)].map((m: any) => [m.c.name, m.p.number]), [["ann", "1"], ["bob", "2"]]));
  const Owner = new P.OfPredicate.Builder().name("Owner").symbols({ c: Contact, p: Phone }).requires(
    P.Contains(p.owners, (e) => e.owner.eq(c))).create();
  assert(same(query.explain(Owner), ["p: scan Phone", "c: p.owners to owner, at most one, then 1 test"]));
  const Unowned = new P.OfPredicate.Builder().name("Unowned").symbols({ p: Phone }).forbids(
    P.Exists((q) => q.symbols({ c: Contact }).requires(P.Contains(c.phones, (e) => e.phone.eq(p))))).create();
  assert([...query.select(Unowned)].length === 0 && [...query.select(Unowned, null, true)].length === 0);
  // A relation of more than two links: the condition names the link.
  const Assigned = new S.OfRelation.Builder().name("Assigned").links("desk", "person", "seat").create();
  const Person = new S.OfObject.Builder().name("Person").ref().relations((r) => r.name("desks").of(Assigned).me("person")).create();
  const Seat = new S.OfObject.Builder().name("Seat").ref().relations((r) => r.name("desks").of(Assigned).me("seat")).create();
  const Desk = new S.OfObject.Builder().name("Desk").ref().singleton("office.Desk").relations(
    (r) => r.name("assigned").of(Assigned).me("desk")).create();
  const office: any = new Proxies.OfStore();
  for (const schema of [Assigned, Person, Seat, Desk]) office.register(schema);
  const [desk, pam, seat] = [office.singleton("office.Desk"), office.Person().create(), office.Seat().create()];
  office.Desk(desk).assigned((e: any) => e.person(pam).seat(seat)).update();
  const d = E.variable("d");
  const Seated = new P.OfPredicate.Builder().name("Seated").symbols({ d: Desk, x: Person }).requires(
    P.Contains(d.assigned, (e) => e.person.eq(x))).create();
  assert(same(new Q.Scan(office).explain(Seated), ["d: scan Desk", "x: d.assigned to person, any number, then 1 test"]));
  assert(equal([...new Q.Scan(office).select(Seated)], [{ d: desk, x: pam }]));
  const predicates = new C.OfSet.Builder().predicates(EveryoneHasAPhone, Numbered, (b) => b.name("Possible").requires(some)).create();
  const schemas = new C.OfStore(store);
  const text = SchemaJSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates);
  assert(text.includes('"$schema": "Patterns.OfForall"') && text.includes('"$schema": "Patterns.OfChoices"'));
  const copy = SchemaJSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text) as C.OfSet.Data;
  assert(copy.predicates.every((a, i) => Terms.same(a.rule, predicates.predicates[i]!.rule)));
  assert((copy.predicates[2]!.rule as any).body.arms[0].weight === 0.35);
}